# 04-9. 파일 분석기 종합 실습

## Goal

이 노트북은 `examples/04-file-analyzer/file_analyzer.py` 기준 구현을 복제하지 않고 안전하게 불러와 인수 조건을 검증하는 풀이 확인용 자료입니다. 다음 내용을 작은 합성 파일로 확인합니다.

- 원본을 변경하지 않는 공통 분석과 SHA-256 불변성
- 텍스트·CSV·JSON 통계와 복구 가능한 형식 오류 보존
- 매직 바이트와 확장자 불일치, 빈 파일 정책
- 보고서 구조 검증과 임시 파일을 이용한 안전한 저장
- 존재하지 않는 파일·디렉터리·전체 UTF-8 오류의 실패 경계

## Setup

04장 교육자료 ZIP을 풀고 `notebooks`와 `examples` 폴더를 함께 유지합니다. 현재 작업 디렉터리부터 상위 디렉터리를 확인해 `examples/04-file-analyzer/file_analyzer.py`가 있는 실습 루트를 찾습니다. 따라서 `notebooks` 폴더를 기준으로 실행하는 커널에서도 예제 모듈을 찾을 수 있습니다. 하이픈이 있는 예제 디렉터리는 `importlib`로 파일 경로에서 로드합니다. 모듈 이름이 `__main__`이 아니므로 대화형 `main()`은 실행되지 않습니다.

모든 fixture와 보고서는 `TemporaryDirectory` 안에 만들며, 실제 문서·운영 로그·실행 파일은 사용하지 않습니다.

In [1]:
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

assert sys.version_info >= (3, 10), "Python 3.10 이상이 필요합니다"
sys.dont_write_bytecode = True

def find_project_root() -> Path:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'examples/04-file-analyzer/file_analyzer.py').is_file():
            return candidate
    raise FileNotFoundError('04장 자료 ZIP을 풀고 notebooks와 examples 폴더를 함께 유지하세요.')


PROJECT_ROOT = find_project_root()
MODULE_PATH = (
    PROJECT_ROOT
    / "examples"
    / "04-file-analyzer"
    / "file_analyzer.py"
)
if not MODULE_PATH.is_file():
    raise FileNotFoundError(
        "저장소 루트에서 Jupyter를 실행하세요. "
        f"찾지 못한 파일: {MODULE_PATH}"
    )

spec = importlib.util.spec_from_file_location(
    "chapter04_file_analyzer",
    MODULE_PATH,
)
if spec is None or spec.loader is None:
    raise ImportError(f"모듈 로더를 만들 수 없습니다: {MODULE_PATH}")

file_analyzer = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = file_analyzer
spec.loader.exec_module(file_analyzer)
assert file_analyzer.__name__ != "__main__"

_lab = TemporaryDirectory(prefix="chapter-04-9-")
LAB_DIR = Path(_lab.name).resolve()

print("프로젝트 루트:", PROJECT_ROOT)
print("불러온 구현:", MODULE_PATH)
print("임시 실습 디렉터리:", LAB_DIR)

프로젝트 루트: /Users/jeongseongmin/Documents/ChatGPT/화이트해커 양성 - Python 교안/Python
불러온 구현: /Users/jeongseongmin/Documents/ChatGPT/화이트해커 양성 - Python 교안/Python/examples/04-file-analyzer/file_analyzer.py
임시 실습 디렉터리: /private/var/folders/yh/sdghdz7x60s7sjvhdyv7d_m80000gn/T/chapter-04-9-nvzha979


## Steps

### 1. 무해하고 재현 가능한 fixture 만들기

교안의 기본 6개 입력에 JSON 정책 오류, CSV 구조 오류, 표본 뒤의 UTF-8 오류를 추가합니다. 모두 학습자가 직접 생성하는 작은 데이터이며 파일로 실행하거나 외부로 전송하지 않습니다.

In [2]:
def create_fixtures(folder: Path) -> dict[str, Path]:
    fixtures = {
        "sample.txt": "첫째 줄\n\n셋째 줄\n",
        "sample.csv": "name,score\n민준,90\n서연,\n",
        "sample.json": '{"name": "학습 보고서", "count": 2}\n',
        "broken.json": '{"name": "닫히지 않은 JSON"\n',
        "duplicate.json": '{"name": "첫 값", "name": "둘째 값"}\n',
        "nonstandard.json": '{"value": NaN}\n',
        "broken.csv": 'name,note\n민준,"닫히지 않은 값\n',
    }
    for name, text in fixtures.items():
        (folder / name).write_text(text, encoding="utf-8")

    (folder / "renamed.txt").write_bytes(
        b"%PDF-1.7\ntraining sample\n"
    )
    (folder / "empty.bin").write_bytes(b"")
    (folder / "late-invalid.txt").write_bytes(
        b"a" * file_analyzer.SAMPLE_BYTES + b"\xff"
    )

    return {path.name: path for path in folder.iterdir() if path.is_file()}


INPUT_DIR = LAB_DIR / "input"
INPUT_DIR.mkdir()
fixtures = create_fixtures(INPUT_DIR)
assert all(path.is_relative_to(LAB_DIR) for path in fixtures.values())
print("fixture:", sorted(fixtures))

fixture: ['broken.csv', 'broken.json', 'duplicate.json', 'empty.bin', 'late-invalid.txt', 'nonstandard.json', 'renamed.txt', 'sample.csv', 'sample.json', 'sample.txt']


### 2. 공통 분석과 원본 보호 확인하기

`analyze_file()`은 보고서를 딕셔너리로 반환할 뿐 파일을 쓰지 않습니다. 분석 전후의 SHA-256을 비교해 CSV 원본이 바뀌지 않았는지 확인합니다.

In [3]:
csv_target = fixtures["sample.csv"]
source_hash_before = file_analyzer.calculate_sha256(csv_target)
csv_report = file_analyzer.analyze_file(csv_target)
source_hash_after_analysis = file_analyzer.calculate_sha256(csv_target)

common_view = {
    key: csv_report[key]
    for key in (
        "name",
        "suffix",
        "size_bytes",
        "sha256",
        "content_type",
        "encoding",
        "file_type",
    )
}
print(json.dumps(common_view, ensure_ascii=False, indent=2))

{
  "name": "sample.csv",
  "suffix": ".csv",
  "size_bytes": 29,
  "sha256": "61149f2ffd9ba83c8c8972f021a9a23f229a4fac95460580410039f1b67c9c7f",
  "content_type": "text",
  "encoding": "utf-8",
  "file_type": {
    "detected_format": "unknown",
    "mime": "application/octet-stream",
    "expected_extensions": [],
    "extension_matches": null,
    "signature_offset": null,
    "header_hex": "6e 61 6d 65 2c 73 63 6f 72 65 0a eb af bc ec a4",
    "match": {
      "status": "NO",
      "reason": null
    }
  }
}


### 3. 텍스트·CSV·JSON과 형식 오류 비교하기

전체 UTF-8 읽기 실패를 의도한 `late-invalid.txt`를 제외하고 각 fixture를 분석합니다. CSV·JSON의 복구 가능한 형식 오류는 예외로 중단되지 않고 공통 보고서의 `format_error`에 남습니다.

In [4]:
report_names = [
    "sample.txt",
    "sample.csv",
    "sample.json",
    "broken.json",
    "duplicate.json",
    "nonstandard.json",
    "broken.csv",
    "renamed.txt",
    "empty.bin",
]
reports = {
    name: file_analyzer.analyze_file(fixtures[name])
    for name in report_names
}

overview = []
for name in report_names:
    report = reports[name]
    format_name = next(iter(report.get("format", {})), None)
    overview.append({
        "name": name,
        "content_type": report["content_type"],
        "detected_format": report["file_type"]["detected_format"],
        "format": format_name,
        "format_error": report.get("format_error", {}).get("type"),
    })

print(json.dumps(overview, ensure_ascii=False, indent=2))

[
  {
    "name": "sample.txt",
    "content_type": "text",
    "detected_format": "unknown",
    "format": null,
    "format_error": null
  },
  {
    "name": "sample.csv",
    "content_type": "text",
    "detected_format": "unknown",
    "format": "csv",
    "format_error": null
  },
  {
    "name": "sample.json",
    "content_type": "text",
    "detected_format": "unknown",
    "format": "json",
    "format_error": null
  },
  {
    "name": "broken.json",
    "content_type": "text",
    "detected_format": "unknown",
    "format": null,
    "format_error": "JSONDecodeError"
  },
  {
    "name": "duplicate.json",
    "content_type": "text",
    "detected_format": "unknown",
    "format": null,
    "format_error": "JSONPolicyError"
  },
  {
    "name": "nonstandard.json",
    "content_type": "text",
    "detected_format": "unknown",
    "format": null,
    "format_error": "JSONPolicyError"
  },
  {
    "name": "broken.csv",
    "content_type": "text",
    "detected_format": "unknown",


### 4. 분석 보고서를 안전하게 저장하기

CLI와 같은 이름 규칙으로 원본 이름 뒤에 `.analysis.json`을 붙입니다. 출력도 임시 실습 디렉터리 안에 있고 입력과 다른 경로인지 확인한 뒤, 기준 구현의 `save_report()`로 저장하고 다시 읽어 검증합니다.

In [5]:
OUTPUT_DIR = LAB_DIR / "results"
report_path = OUTPUT_DIR / (csv_target.name + ".analysis.json")
assert report_path.is_relative_to(LAB_DIR)
file_analyzer.ensure_different_paths(csv_target, report_path)
file_analyzer.save_report(csv_report, report_path)

with report_path.open("r", encoding="utf-8") as file:
    restored_report = json.load(file)
file_analyzer.validate_report(restored_report)

source_hash_after_save = file_analyzer.calculate_sha256(csv_target)
temporary_report_files = list(
    OUTPUT_DIR.glob(f".{report_path.name}.*.tmp")
)
print("보고서 저장 경로:", report_path)

보고서 저장 경로: /private/var/folders/yh/sdghdz7x60s7sjvhdyv7d_m80000gn/T/chapter-04-9-nvzha979/results/sample.csv.analysis.json


### 5. 전체 작업 실패 경계 확인하기

존재하지 않는 경로와 디렉터리는 분석 함수가 예외로 알립니다. 현재 참고 분석기는 텍스트를 `errors="replace"`로 읽으므로 표본 뒤의 잘못된 UTF-8도 대체 문자로 처리합니다. 따라서 보고서 생성 성공이 원문 전체의 인코딩 유효성을 증명하지는 않습니다. 같은 파일을 `errors="strict"`로 다시 읽어 차이를 확인합니다. 이 노트북은 `main()`을 실행하지 않고 분석·저장 함수를 각각 검증합니다.

In [6]:
def capture_exception(function, *args) -> dict[str, str]:
    try:
        function(*args)
    except Exception as exc:
        return {"type": type(exc).__name__, "message": str(exc)}
    raise AssertionError("예상한 예외가 발생하지 않았습니다")


missing_error = capture_exception(
    file_analyzer.analyze_file,
    LAB_DIR / "missing.txt",
)
directory_error = capture_exception(file_analyzer.analyze_file, LAB_DIR)
def read_strict_utf8(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="strict")


late_report = file_analyzer.analyze_file(fixtures["late-invalid.txt"])
late_encoding_error = capture_exception(
    read_strict_utf8,
    fixtures["late-invalid.txt"],
)
invalid_report = dict(csv_report)
invalid_report.pop("sha256")
report_contract_error = capture_exception(
    file_analyzer.validate_report,
    invalid_report,
)

print(json.dumps({
    "missing": missing_error,
    "directory": directory_error,
    "late_encoding": late_encoding_error,
    "invalid_report": report_contract_error,
}, ensure_ascii=False, indent=2))

{
  "missing": {
    "type": "FileNotFoundError",
    "message": "파일이 없습니다: /private/var/folders/yh/sdghdz7x60s7sjvhdyv7d_m80000gn/T/chapter-04-9-nvzha979/missing.txt"
  },
  "directory": {
    "type": "ValueError",
    "message": "일반 파일이 아닙니다: /private/var/folders/yh/sdghdz7x60s7sjvhdyv7d_m80000gn/T/chapter-04-9-nvzha979"
  },
  "late_encoding": {
    "type": "UnicodeDecodeError",
    "message": "'utf-8' codec can't decode byte 0xff in position 8192: invalid start byte"
  },
  "invalid_report": {
    "type": "ValueError",
    "message": "보고서 필수 필드가 없습니다: sha256"
  }
}


## Checks

교안의 테스트 매트릭스와 추가 정책 오류를 `assert`로 검증합니다. 하나라도 실패하면 보고서 해석이나 실제 파일 분석으로 진행하지 않습니다.

In [7]:
assert source_hash_before == source_hash_after_analysis == source_hash_after_save
assert len(csv_report["sha256"]) == 64

text_report = reports["sample.txt"]
assert text_report["content_type"] == "text"
assert text_report["text"]["line_count"] == 3
assert text_report["text"]["blank_line_count"] == 1

csv_stats = reports["sample.csv"]["format"]["csv"]
assert csv_stats["header"] == ["name", "score"]
assert csv_stats["data_row_count"] == 2
assert csv_stats["rows_with_missing_values"] == 1
assert csv_stats["rows_with_wrong_column_count"] == 0

json_stats = reports["sample.json"]["format"]["json"]
assert json_stats["top_level_type"] == "dict"
assert json_stats["top_level_key_count"] == 2

assert reports["broken.json"]["format_error"]["type"] == "JSONDecodeError"
assert reports["duplicate.json"]["format_error"]["type"] == "JSONPolicyError"
assert reports["nonstandard.json"]["format_error"]["type"] == "JSONPolicyError"
assert reports["broken.csv"]["format_error"]["type"] == "Error"

renamed_header = reports["renamed.txt"]["file_type"]
assert renamed_header["detected_format"] == "PDF document"
assert renamed_header["extension_matches"] is False
assert renamed_header["match"]["status"] == "NO"
assert renamed_header["match"]["reason"]

empty_report = reports["empty.bin"]
assert empty_report["size_bytes"] == 0
assert empty_report["content_type"] == "text"
assert empty_report["text"]["line_count"] == 0

assert restored_report == csv_report
assert report_path.name == "sample.csv.analysis.json"
assert temporary_report_files == []
assert missing_error["type"] == "FileNotFoundError"
assert directory_error["type"] == "ValueError"
assert late_encoding_error["type"] == "UnicodeDecodeError"
assert late_report["content_type"] == "text"
assert late_report["text"]["line_count"] == 1
assert report_path.parent != csv_target.parent
assert csv_report["mutation"]["detected"] is False
assert report_contract_error["type"] == "ValueError"
assert "sha256" in report_contract_error["message"]

print("04-9 분석기 계약 검증 통과")

04-9 분석기 계약 검증 통과


In [8]:
temporary_root = LAB_DIR
_lab.cleanup()
assert not temporary_root.exists()
print("임시 실습 디렉터리 정리 완료")

임시 실습 디렉터리 정리 완료


## Next Steps

- 실제 파일을 분석하기 전 허가된 작업 디렉터리, 최대 파일 크기, 한 레코드의 최대 길이를 정책으로 정합니다.
- UTF-16이나 표본 경계에서 잘린 UTF-8처럼 휴리스틱이 오분류할 수 있는 입력을 별도 과제로 검증합니다.
- 여러 파일을 처리할 때는 파일별 성공·오류·건너뜀을 분리하고 전체 건수의 보존 법칙을 추가합니다.
- 헤더·해시는 형식과 내용 비교 신호일 뿐 파일의 안전성·출처·작성자를 증명하지 않습니다.